# StudyLens: final grounding evaluation on fresh AI2D test diagrams

Runs `scripts/evaluate_grounding.py --tag v2d` (scene graph with the trained **arrow detector**, unreliable
boxes hidden) on **100 AI2D test diagrams that were never used during development**: the 100 dev diagrams
are left out with `--exclude dev_ids.txt`. Same sample rule and seed (458) as the dev run: 60 "parts of"
diagrams and 40 flow diagrams, up to 3 questions each (127 questions).

* **GPU:** a **T4** is enough (Qwen2.5-VL-3B in fp16 uses about 8 GB). **Time:** a few hours (5.5 h on the M4 Mac).
* **Resumable:** every model answer is cached on Drive as it is produced. If Colab disconnects, run the
  notebook again from the top; finished diagrams are skipped.
* **You need on Drive** (in `MyDrive/studylens/`), from your Mac:
  * `checkpoints/arrow_detector.pt` (already there from training)
  * `checkpoints/grouping_model.json` (the learned text grouping, 2 KB)
  * `grounding_runs/dev_ids.txt` (the 100 dev diagram ids)
* **Licences:** AI2D is non-commercial; the raw answers (which quote AI2D text) stay on your Drive. The
  results file holds ids and numbers only.
* **fp16 note:** the Mac runs Qwen in bf16 and the T4 in fp16, so a few answers can differ slightly from a
  Mac run. Report the platform with the numbers.

## 1. Drive and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/studylens'
!mkdir -p $DRIVE/external $DRIVE/checkpoints $DRIVE/grounding_runs
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Check the files from your Mac
Upload any that are missing with the Files panel on the left (or drag them into Drive in the browser).
On the Mac they are `checkpoints/grouping_model.json` and `external/grounding_runs/dev_ids.txt` in the repo.

In [ ]:
import os
need = {
    'arrow detector weights': f'{DRIVE}/checkpoints/arrow_detector.pt',
    'learned text grouping':  f'{DRIVE}/checkpoints/grouping_model.json',
    'dev diagram ids':        f'{DRIVE}/grounding_runs/dev_ids.txt',
}
missing = [k for k, p in need.items() if not os.path.exists(p)]
for k, p in need.items():
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + k.ljust(24) + p)
n_dev = len(open(need['dev diagram ids']).read().split()) if os.path.exists(need['dev diagram ids']) else 0
print('dev ids:', n_dev)
assert not missing, 'upload the missing files, then run this cell again'
assert n_dev == 100, 'dev_ids.txt should list 100 diagrams'

## 3. Code and packages (2-4 min)

In [ ]:
REPO_URL = 'https://github.com/krish-patel1003/CECS558-term-project-studylens.git'
if not os.path.exists('/content/studylens'):
    !git clone -q $REPO_URL /content/studylens
%cd /content/studylens
!git pull -q
!git log --oneline -1
!pip install -q -r requirements-model.txt

## 4. OCR in its own environment (3-5 min)
PaddleOCR's version pins conflict with PyTorch, so (as on the Mac) it runs in a separate Python
environment that StudyLens talks to through a small worker process. It runs on the CPU.

In [ ]:
!rm -rf external checkpoints        # removes old symlinks only, not the Drive folders
!mkdir -p external
!ln -s $DRIVE/checkpoints checkpoints
if not os.path.exists('/content/venv-ocr/bin/python'):
    !pip install -q virtualenv
    !virtualenv -q /content/venv-ocr
    !/content/venv-ocr/bin/pip install -q -r requirements-ocr.txt
os.environ['OCR_PYTHON'] = '/content/venv-ocr/bin/python'
# smoke test: the worker must start and read text
!OCR_PYTHON=/content/venv-ocr/bin/python python -c "from PIL import Image, ImageDraw; from studylens.ocr import get_ocr; im = Image.new('RGB', (360, 90), 'white'); ImageDraw.Draw(im).text((20, 30), 'mitochondria', fill='black', font_size=36); o = get_ocr(); print('OCR:', [t.get('text') for t in o.read(im)]); o.close()"

## 5. AI2D on the local disk; the answer cache on Drive
AI2D is copied from Drive (it is there from the detector training) or downloaded. The cache
`grounding_runs/grounding_cache.jsonl` lives on Drive so a disconnect loses at most one diagram.

In [ ]:
if os.path.exists(f'{DRIVE}/external/ai2d/ai2d/categories.json'):
    !rsync -a --info=progress2 $DRIVE/external/ai2d/ external/ai2d/
else:
    !python scripts/get_ai2d.py --accept-license
    !rsync -a external/ai2d/ $DRIVE/external/ai2d/
!ln -s $DRIVE/grounding_runs external/grounding_runs
!ls external/ai2d/ai2d/images | wc -l

# a disconnect can leave a half-written last line in the cache: drop it so the run can resume
cache = f'{DRIVE}/grounding_runs/grounding_cache.jsonl'
if os.path.exists(cache):
    import json
    lines = open(cache).read().splitlines()
    good = []
    for l in lines:
        try:
            json.loads(l); good.append(l)
        except ValueError:
            pass
    if len(good) != len(lines):
        open(cache, 'w').write('\n'.join(good) + '\n')
    print(f'cache: {len(good)} answers already done ({len(lines) - len(good)} broken lines dropped)')

## 6. Check the sample (no model calls)
Must say **100 diagrams, 127 questions**, with **0** overlap with the dev set.

In [ ]:
import sys
sys.path.insert(0, 'scripts')
import evaluate_grounding as eg
dev = set(open('external/grounding_runs/dev_ids.txt').read().split())
diagrams, items, _ = eg.sample(60, 40, 458, 3, dev)
print(len(diagrams), 'diagrams,', len(items), 'questions; overlap with dev:', len(dev & {d.image_id for d in diagrams}))
!ls -la checkpoints/arrow_detector.pt checkpoints/grouping_model.json

## 7. Run the evaluation (progress bar)
The script runs in a worker process; this cell shows a progress bar with the time left and keeps the full
log in `grounding_runs/v2d_test.log` on Drive. Keep the tab open. If the session drops, reconnect and run
everything again from the top: it resumes from the cache.

In [ ]:
import re, subprocess, time
from tqdm.auto import tqdm

def run_eval(tag, log_name):
    cmd = ['python', '-u', 'scripts/evaluate_grounding.py', '--tag', tag, '--n-parts', '60', '--n-flow', '40',
           '--exclude', 'external/grounding_runs/dev_ids.txt',
           '--write-ids', 'external/grounding_runs/test_ids.txt']
    env = dict(os.environ, OCR_PYTHON='/content/venv-ocr/bin/python', TOKENIZERS_PARALLELISM='false')
    log = open(f'external/grounding_runs/{log_name}', 'a')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    bar, tail = None, []
    for line in proc.stdout:
        log.write(line); log.flush()
        m = re.match(r'\[(\d+)/(\d+)\] (\S+) \((\w+)\)', line)
        if m:
            done, total = int(m.group(1)), int(m.group(2))
            if bar is None:
                bar = tqdm(total=total, desc=f'{tag} diagrams', unit='diagram')
            bar.n = done; bar.set_postfix_str(f'{m.group(3)} ({m.group(4)})'); bar.refresh()
        elif re.match(r'\d+ diagrams, \d+ questions', line) or 'Error' in line or 'Traceback' in line:
            print(line, end='')
        tail = (tail + [line])[-60:]
    proc.wait()
    if bar: bar.close()
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'evaluation stopped (exit {proc.returncode}); see the log on Drive, then run this cell again')
    print('finished')

t0 = time.time()
run_eval('v2d', 'v2d_test.log')
print(f'{(time.time() - t0) / 3600:.1f} h')

## 8. Results
Saved as `results/grounding_eval_v2d_x3_test.json` (ids and numbers only) and copied to Drive. Download it
and put it in the repo's `results/` folder.

In [ ]:
import json, shutil
R = 'results/grounding_eval_v2d_x3_test.json'
shutil.copy(R, f'{DRIVE}/grounding_runs/')
s = json.load(open(R))['summary']
o, b = s['pointing']['OURS'], s['pointing']['B2']
mc, fl = s['multiple_choice'], s['flow_reference_precision']
rows = [
    ('Pointing accuracy',      f"{o['pointing_accuracy']:.1%} ({o['hits']}/{o['matched']})", f"{b['pointing_accuracy']:.1%} ({b['hits']}/{b['matched']})"),
    ('Coverage',               f"{o['coverage']:.1%} ({o['covered']}/{o['gold']})" if 'gold' in o else f"{o['coverage']:.1%}", f"{b['coverage']:.1%} ({b['covered']}/{b['gold']})"),
    ('Cited arrows that are real', f"{fl['precision']:.1%} ({fl['gold_arrows']}/{fl['cited_arrows']})", 'n/a'),
    ('Multiple choice',        f"E3 {mc['E3']['accuracy']:.1%}  vs  E2 {mc['E2']['accuracy']:.1%}  (p = {mc['E2_vs_E3']['p_value']})", ''),
    ('Answered by the graph',  str(mc['answered_by_query']), ''),
]
print(f"{'':28s}{'StudyLens (v2d x3)':32s}B2")
for r in rows:
    print(f'{r[0]:28s}{r[1]:32s}{r[2]}')
print('\nDev set, for comparison: pointing 68.5% vs B2 65.3%; coverage 32.2% vs 36.7%; cited arrows real 65.8%; MC 94.4%')

In [ ]:
from google.colab import files
files.download(R)

## 9. Optional: the same test diagrams with the VLM's arrow list (`v2`)
Gives the before/after comparison for the arrow detector on the test set. B2 and E2 answers are reused from
the cache, so this takes roughly half as long as step 7. Output: `results/grounding_eval_v2_x3_test.json`.

In [ ]:
run_eval('v2', 'v2_test.log')
R2 = 'results/grounding_eval_v2_x3_test.json'
shutil.copy(R2, f'{DRIVE}/grounding_runs/')
files.download(R2)